In [2]:
import sys
from pathlib import Path

ROOT = Path.cwd().parents[1]
sys.path.append(str(ROOT))

import pandas as pd

from src.feature_selection.wrapper import rfe_random_forest

In [6]:
mrmr_path = "../../data/processed/filter_results/mrmr_results.csv"

mrmr_results = pd.read_csv(mrmr_path)

mrmr_results

,rank,feature
0,1,sma_5
1,2,volume_change_1d
2,3,price_to_sma_5
3,4,roc_20
4,5,volume_ratio_20
5,6,volatility_5
6,7,gap
7,8,high_low_range
8,9,rsi_14
9,10,intraday_return


In [7]:
mrmr_features = mrmr_results["feature"].tolist()

mrmr_features

['sma_5',
 'volume_change_1d',
 'price_to_sma_5',
 'roc_20',
 'volume_ratio_20',
 'volatility_5',
 'gap',
 'high_low_range',
 'rsi_14',
 'intraday_return']

In [9]:
data_path = ROOT / "data/processed/ml_dataset.csv"

df = pd.read_csv(data_path)

df.head()

,stock_code,trade_date,return_1d,return_5d,return_10d,return_20d,intraday_return,high_low_range,gap,sma_5,...,macd,macd_signal,macd_hist,volatility_5,volatility_20,atr_14,volume_change_1d,volume_sma_20,volume_ratio_20,target_return_5d
0,660,2024-06-11,0.021635,0.094233,0.054591,0.181212,0.011905,0.043689,0.009615,203000.0,...,6187.902397,5195.016563,992.885834,0.031048,0.023996,6757.468051,0.253659,3435519.80,0.893653,0.103529
1,660,2024-06-12,0.011765,0.112261,0.061728,0.169750,0.014151,0.023697,-0.002353,207340.0,...,6911.664995,5538.346249,1373.318746,0.028770,0.023814,6631.934619,-0.299278,3381585.20,0.636190,0.086047
2,660,2024-06-13,0.032558,0.146102,0.096296,0.198057,-0.017699,0.034247,0.051163,213000.0,...,7958.354609,6022.347921,1936.006687,0.026692,0.024432,6979.653575,1.685444,3532295.70,1.635559,0.069820
3,660,2024-06-14,-0.004505,0.065060,0.129280,0.145078,-0.017778,0.041667,0.013514,215700.0,...,8607.944994,6539.467336,2068.477659,0.014806,0.023386,7123.964034,-0.426854,3443697.95,0.961531,0.058824
4,660,2024-06-17,0.009050,0.072115,0.178647,0.174302,0.018265,0.047945,-0.009050,218700.0,...,9178.331251,7067.240119,2111.091132,0.013915,0.022745,7365.109460,-0.336094,3411545.25,0.644382,0.000000


In [11]:
# 날짜 타입 변환
df["trade_date"] = pd.to_datetime(df["trade_date"])

# Train / Validation 분할
train_df = df[
    (df["trade_date"] >= "2024-03-13") &
    (df["trade_date"] <= "2025-12-31")
].copy()

valid_df = df[
    (df["trade_date"] >= "2026-01-01") &
    (df["trade_date"] <= "2026-06-30")
].copy()

# Target
target = "return_5d"

print("train:", train_df.shape)
print("validation:", valid_df.shape)
print("target:", target)

train: (1895, 28)
validation: (600, 28)
target: return_5d


In [12]:
missing_features = [
    f for f in mrmr_features
    if f not in train_df.columns
]

print("MRMR features:", mrmr_features)
print("Missing features:", missing_features)

MRMR features: ['sma_5', 'volume_change_1d', 'price_to_sma_5', 'roc_20', 'volume_ratio_20', 'volatility_5', 'gap', 'high_low_range', 'rsi_14', 'intraday_return']
Missing features: []


In [14]:
rfe_rf_results = rfe_random_forest(
    train_df=train_df,
    valid_df=valid_df,
    features=mrmr_features,
    target=target,
)

print("RFE + Random Forest 완료")
print(rfe_rf_results)

RFE + Random Forest 완료
                method  n_features  \
0  RFE + Random Forest           1   
1  RFE + Random Forest           2   
2  RFE + Random Forest           3   
3  RFE + Random Forest           4   
4  RFE + Random Forest           5   
5  RFE + Random Forest           6   
6  RFE + Random Forest           7   
7  RFE + Random Forest           8   
8  RFE + Random Forest           9   
9  RFE + Random Forest          10   

                                            features      rmse  
0                                   [price_to_sma_5]  0.064681  
1                           [price_to_sma_5, rsi_14]  0.056847  
2             [price_to_sma_5, volatility_5, rsi_14]  0.058030  
3     [price_to_sma_5, roc_20, volatility_5, rsi_14]  0.058135  
4  [price_to_sma_5, roc_20, volatility_5, rsi_14,...  0.056423  
5  [price_to_sma_5, roc_20, volatility_5, gap, rs...  0.056458  
6  [sma_5, price_to_sma_5, roc_20, volatility_5, ...  0.056806  
7  [sma_5, price_to_sma_5, roc_20, vol

In [15]:
best_rfe_rf = rfe_results.loc[rfe_results["rmse"].idxmin()]

print("Best RFE + Random Forest")
print("n_features:", best_rfe_rf["n_features"])
print("RMSE:", best_rfe_rf["rmse"])
print("features:", best_rfe_rf["features"])

Best RFE + Random Forest
n_features: 5
RMSE: 0.05642265944041948
features: ['price_to_sma_5', 'roc_20', 'volatility_5', 'rsi_14', 'intraday_return']
